# Pharmacokinetic/Pharmacodynamic (PK/PD) Modeling with Deep Learning

This notebook covers PK/PD modeling — the study of how drugs move through and affect the body — using both classical compartmental approaches and neural network-based methods with TensorFlow.

## Contents
1. **Background** — PK/PD fundamentals
2. **Classical Compartmental Models** — One- and two-compartment PK models via ODEs
3. **Simulating PK Data** — Generating synthetic concentration-time profiles
4. **Neural Network PK Parameter Estimation** — Using a neural network to predict PK parameters from noisy concentration-time data
5. **Neural ODE for PK Modeling** — Approximating drug dynamics with a learned ODE
6. **PD Modeling** — Linking drug concentration to pharmacological effect (Emax model)
7. **Full PK/PD Pipeline** — End-to-end deep learning model for PK/PD prediction

In [1]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")

I0000 00:00:1774688675.926976    3280 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774688675.927588    3280 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774688675.981544    3280 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.21.0
NumPy version: 2.4.3


I0000 00:00:1774688677.455339    3280 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774688677.455735    3280 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## 1. Background

**Pharmacokinetics (PK)** describes *what the body does to the drug*:
- **Absorption** — drug enters systemic circulation
- **Distribution** — drug spreads to tissues
- **Metabolism** — drug is chemically transformed (primarily in the liver)
- **Elimination** — drug is removed from the body

Key PK parameters:
- **CL** (Clearance): volume of plasma cleared of drug per unit time (L/h)
- **Vd** (Volume of distribution): apparent volume relating plasma concentration to total drug (L)
- **ka** (Absorption rate constant): rate of drug absorption from the gut (1/h)
- **ke** (Elimination rate constant): ke = CL / Vd (1/h)
- **t½** (Half-life): time for concentration to halve; t½ = ln(2) / ke

**Pharmacodynamics (PD)** describes *what the drug does to the body*:
- **Emax model**: Effect = (Emax × C) / (EC50 + C)
- **Emax**: maximum possible effect
- **EC50**: concentration producing 50% of Emax

## 2. Classical Compartmental PK Models

### One-Compartment Model (IV Bolus)

The simplest PK model assumes the body acts as a single well-mixed compartment:

$$\frac{dC}{dt} = -k_e \cdot C$$

Solution: $C(t) = C_0 \cdot e^{-k_e \cdot t}$ where $C_0 = \text{Dose} / V_d$

### One-Compartment Model (Oral Dosing)

With first-order absorption from the gut:

$$\frac{dA_g}{dt} = -k_a \cdot A_g$$
$$\frac{dC}{dt} = \frac{k_a \cdot A_g}{V_d} - k_e \cdot C$$

### Two-Compartment Model (IV Bolus)

Adds a peripheral tissue compartment:

$$\frac{dC_1}{dt} = -\left(\frac{CL + Q}{V_1}\right) C_1 + \frac{Q}{V_2} C_2$$
$$\frac{dC_2}{dt} = \frac{Q}{V_1} C_1 - \frac{Q}{V_2} C_2$$

where Q is the inter-compartmental clearance, V1 is central volume, V2 is peripheral volume.

In [2]:
# --- One-Compartment PK Models ---

def one_compartment_iv(dose, vd, ke, t):
    """Analytical solution for 1-compartment IV bolus."""
    c0 = dose / vd
    return c0 * np.exp(-ke * t)

def one_compartment_oral_ode(y, t, ka, ke, vd):
    """ODE system for 1-compartment oral dosing."""
    a_gut, c_plasma = y
    da_gut_dt = -ka * a_gut
    dc_plasma_dt = (ka * a_gut) / vd - ke * c_plasma
    return [da_gut_dt, dc_plasma_dt]

def two_compartment_iv_ode(y, t, cl, q, v1, v2):
    """ODE system for 2-compartment IV bolus."""
    c1, c2 = y
    dc1_dt = -(cl + q) / v1 * c1 + q / v2 * c2
    dc2_dt = q / v1 * c1 - q / v2 * c2
    return [dc1_dt, dc2_dt]

# Parameters
dose = 100       # mg
vd = 50          # L
cl = 5           # L/h
ke = cl / vd     # 0.1 /h
ka = 1.5         # /h (absorption rate)
q = 2            # L/h (inter-compartmental clearance)
v1 = 20          # L (central volume)
v2 = 40          # L (peripheral volume)

t = np.linspace(0, 24, 500)

# 1-compartment IV
c_iv = one_compartment_iv(dose, vd, ke, t)

# 1-compartment oral
y0_oral = [dose, 0]  # drug starts in gut
sol_oral = odeint(one_compartment_oral_ode, y0_oral, t, args=(ka, ke, vd))
c_oral = sol_oral[:, 1]

# 2-compartment IV
c0_2comp = dose / v1
y0_2comp = [c0_2comp, 0]
sol_2comp = odeint(two_compartment_iv_ode, y0_2comp, t, args=(cl, q, v1, v2))
c_central = sol_2comp[:, 0]
c_peripheral = sol_2comp[:, 1]

# Plot all models
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(t, c_iv, 'b-', linewidth=2)
axes[0].set_title('1-Compartment IV Bolus')
axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Concentration (mg/L)')
axes[0].set_ylim(bottom=0)
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, c_oral, 'r-', linewidth=2)
axes[1].set_title('1-Compartment Oral Dosing')
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Concentration (mg/L)')
axes[1].set_ylim(bottom=0)
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, c_central, 'g-', linewidth=2, label='Central')
axes[2].plot(t, c_peripheral, 'm--', linewidth=2, label='Peripheral')
axes[2].set_title('2-Compartment IV Bolus')
axes[2].set_xlabel('Time (h)')
axes[2].set_ylabel('Concentration (mg/L)')
axes[2].legend()
axes[2].set_ylim(bottom=0)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PK Parameters:")
print(f"  Half-life (1-comp): {np.log(2)/ke:.1f} h")
print(f"  Tmax (oral, approx): {np.log(ka/ke)/(ka - ke):.1f} h")
print(f"  Cmax (oral): {max(c_oral):.2f} mg/L")

PK Parameters:
  Half-life (1-comp): 6.9 h
  Tmax (oral, approx): 1.9 h
  Cmax (oral): 1.65 mg/L


## 3. Simulating PK Data with Inter-Individual Variability

In real clinical studies, PK parameters vary between patients. We simulate a virtual population using log-normal distributions for PK parameters (standard in population PK modeling).

In [3]:
def simulate_population_pk(n_subjects, n_timepoints, dose, t_max,
                           ka_pop=1.5, ke_pop=0.1, vd_pop=50,
                           omega_ka=0.3, omega_ke=0.3, omega_vd=0.2,
                           sigma_prop=0.1):
    """
    Simulate a population PK dataset with inter-individual variability (IIV)
    and proportional residual error.

    Parameters are drawn from log-normal distributions:
        param_i = param_pop * exp(eta_i), where eta_i ~ N(0, omega^2)
    """
    t = np.linspace(0.5, t_max, n_timepoints)  # start at 0.5h to avoid t=0

    all_profiles = []
    all_params = []
    all_noisy_profiles = []

    for i in range(n_subjects):
        # Individual parameters (log-normal IIV)
        ka_i = ka_pop * np.exp(np.random.normal(0, omega_ka))
        ke_i = ke_pop * np.exp(np.random.normal(0, omega_ke))
        vd_i = vd_pop * np.exp(np.random.normal(0, omega_vd))

        # Solve ODE for oral 1-compartment
        y0 = [dose, 0]
        sol = odeint(one_compartment_oral_ode, y0, t, args=(ka_i, ke_i, vd_i))
        c_true = sol[:, 1]

        # Add proportional residual error
        c_obs = c_true * (1 + np.random.normal(0, sigma_prop, size=len(t)))
        c_obs = np.maximum(c_obs, 0)  # concentrations can't be negative

        all_profiles.append(c_true)
        all_noisy_profiles.append(c_obs)
        all_params.append([ka_i, ke_i, vd_i])

    return t, np.array(all_profiles), np.array(all_noisy_profiles), np.array(all_params)

# Generate population data
n_subjects = 500
n_timepoints = 48
t_sim, profiles_true, profiles_obs, params_true = simulate_population_pk(
    n_subjects=n_subjects, n_timepoints=n_timepoints, dose=100, t_max=24
)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot a subset of individual profiles
for i in range(min(50, n_subjects)):
    axes[0].plot(t_sim, profiles_obs[i], alpha=0.3, linewidth=0.8)
axes[0].set_title(f'Observed Concentration-Time Profiles (n={min(50, n_subjects)})')
axes[0].set_xlabel('Time (h)')
axes[0].set_ylabel('Concentration (mg/L)')
axes[0].grid(True, alpha=0.3)

# Distribution of PK parameters
param_names = ['ka (1/h)', 'ke (1/h)', 'Vd (L)']
for j, name in enumerate(param_names):
    axes[1].hist(params_true[:, j], bins=30, alpha=0.5, label=name, density=True)
axes[1].set_title('Distribution of Individual PK Parameters')
axes[1].set_xlabel('Parameter Value')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Dataset: {n_subjects} subjects, {n_timepoints} timepoints each")
print(f"Parameter ranges:")
for j, name in enumerate(param_names):
    print(f"  {name}: {params_true[:, j].min():.3f} - {params_true[:, j].max():.3f} "
          f"(mean: {params_true[:, j].mean():.3f})")

Dataset: 500 subjects, 48 timepoints each
Parameter ranges:
  ka (1/h): 0.571 - 3.460 (mean: 1.533)
  ke (1/h): 0.038 - 0.208 (mean: 0.103)
  Vd (L): 25.734 - 90.069 (mean: 51.625)


## 4. Neural Network PK Parameter Estimation

Instead of traditional nonlinear mixed-effects modeling (NONMEM), we train a neural network to predict individual PK parameters (ka, ke, Vd) directly from observed concentration-time profiles.

This approach is useful when:
- Traditional fitting is computationally expensive for large populations
- Real-time parameter estimation is needed (e.g., therapeutic drug monitoring)
- The structural model is known but fitting is slow

In [4]:
# Prepare data for neural network
# Input: noisy concentration-time profiles (normalized)
# Output: PK parameters (log-transformed for better training)

# Normalize inputs
profiles_mean = profiles_obs.mean()
profiles_std = profiles_obs.std()
X = (profiles_obs - profiles_mean) / profiles_std

# Log-transform targets (PK parameters are positive, log-normal)
y = np.log(params_true)

# Train/validation/test split
n_train = int(0.7 * n_subjects)
n_val = int(0.15 * n_subjects)

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")
print(f"Input shape: {X_train.shape[1]} timepoints")
print(f"Output shape: {y_train.shape[1]} parameters (log ka, log ke, log Vd)")

Train: 350, Val: 75, Test: 75
Input shape: 48 timepoints
Output shape: 3 parameters (log ka, log ke, log Vd)


In [5]:
# Build a 1D-CNN + Dense model for PK parameter estimation
# CNNs work well here because concentration-time profiles have local temporal patterns

def build_pk_estimator(n_timepoints, n_params):
    model = keras.Sequential([
        # Reshape for Conv1D: (batch, timepoints, 1)
        keras.layers.Reshape((n_timepoints, 1), input_shape=(n_timepoints,)),

        # Temporal convolutions to extract PK features
        keras.layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        keras.layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        keras.layers.MaxPooling1D(2),
        keras.layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        keras.layers.GlobalAveragePooling1D(),

        # Dense layers for parameter prediction
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(n_params)  # linear output (log-space parameters)
    ])
    return model

model_pk = build_pk_estimator(n_timepoints, 3)
model_pk.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae']
)
model_pk.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
E0000 00:00:1774688678.155179    3280 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 48, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 48, 32)         │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 48, 64)         │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 24, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 39,619 (154.76 KB)

 Trainable params: 39,619 (154.76 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Train the model
history_pk = model_pk.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=1)
    ],
    verbose=0
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_pk.history['loss'], label='Train')
axes[0].plot(history_pk.history['val_loss'], label='Validation')
axes[0].set_title('Loss (MSE in log-space)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_pk.history['mae'], label='Train')
axes[1].plot(history_pk.history['val_mae'], label='Validation')
axes[1].set_title('MAE (log-space)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best val loss: {min(history_pk.history['val_loss']):.4f}")
print(f"Best val MAE: {min(history_pk.history['val_mae']):.4f}")


Epoch 44: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.



Epoch 49: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.



Epoch 54: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.



Epoch 59: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.



Epoch 67: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.



Epoch 72: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.



Epoch 77: ReduceLROnPlateau reducing learning rate to 7.812500371073838e-06.


Best val loss: 0.0362
Best val MAE: 0.1256


In [7]:
# Evaluate on test set
y_pred_log = model_pk.predict(X_test, verbose=0)
y_pred = np.exp(y_pred_log)  # back to original scale
y_true = np.exp(y_test)

# Scatter plots: predicted vs true
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
param_names_short = ['ka (1/h)', 'ke (1/h)', 'Vd (L)']

for j in range(3):
    axes[j].scatter(y_true[:, j], y_pred[:, j], alpha=0.5, s=20)
    lims = [min(y_true[:, j].min(), y_pred[:, j].min()) * 0.9,
            max(y_true[:, j].max(), y_pred[:, j].max()) * 1.1]
    axes[j].plot(lims, lims, 'r--', linewidth=1.5, label='Identity')
    axes[j].set_xlabel(f'True {param_names_short[j]}')
    axes[j].set_ylabel(f'Predicted {param_names_short[j]}')
    axes[j].set_title(param_names_short[j])
    axes[j].legend()
    axes[j].grid(True, alpha=0.3)

    # Calculate metrics
    mape = np.mean(np.abs(y_pred[:, j] - y_true[:, j]) / y_true[:, j]) * 100
    r2 = 1 - np.sum((y_true[:, j] - y_pred[:, j])**2) / np.sum((y_true[:, j] - y_true[:, j].mean())**2)
    axes[j].text(0.05, 0.95, f'R² = {r2:.3f}\nMAPE = {mape:.1f}%',
                 transform=axes[j].transAxes, va='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Neural Network PK Parameter Estimation — Test Set', y=1.02)
plt.tight_layout()
plt.show()

## 5. Neural ODE for PK Modeling

Instead of assuming a specific compartmental structure, we can use a neural network to learn the ODE dynamics directly. This is a simplified Neural ODE approach where we train a network to approximate dC/dt = f(C, t).

This is useful when:
- The underlying PK model is unknown or complex
- You want a data-driven model that can capture nonlinear dynamics
- Mechanistic models are too rigid

In [8]:
# Neural ODE approach: learn the derivative function dC/dt = f_nn(C, t)
# We use an Euler integration scheme for simplicity

class NeuralODEModel(keras.Model):
    """
    A neural ODE that learns drug concentration dynamics.
    The network predicts dC/dt given current state [C, t].
    Forward pass integrates using Euler method.
    """
    def __init__(self, hidden_units=64):
        super().__init__()
        # Network that learns the derivative dC/dt
        self.derivative_net = keras.Sequential([
            keras.layers.Dense(hidden_units, activation='tanh'),
            keras.layers.Dense(hidden_units, activation='tanh'),
            keras.layers.Dense(1)  # scalar dC/dt
        ])

    def call(self, inputs):
        """
        inputs: (batch, 2) — [initial_concentration, dose_proxy]
        Returns: predicted concentration profile over time
        """
        c0 = inputs[:, 0:1]          # initial concentration
        dose_info = inputs[:, 1:2]    # dose information

        dt = t_sim[1] - t_sim[0]
        c = c0
        predictions = [c]

        for i in range(len(t_sim) - 1):
            t_norm = tf.constant([[t_sim[i] / 24.0]], dtype=tf.float32)
            t_norm = tf.tile(t_norm, [tf.shape(c)[0], 1])
            state = tf.concat([c, t_norm, dose_info], axis=1)
            dcdt = self.derivative_net(state)
            c = c + dcdt * dt
            c = tf.maximum(c, 0)  # enforce non-negativity
            predictions.append(c)

        return tf.concat(predictions, axis=1)

# Prepare data for Neural ODE
# Use the mean profile for learning dynamics (then extend to population)
mean_profile = profiles_true.mean(axis=0)

# Create training data: use individual profiles
# Input: [C(0), dose] for each subject
c0_values = profiles_obs[:, 0:1]
dose_values = np.full((n_subjects, 1), 100.0 / 50.0)  # dose/vd_pop approximation
X_node = np.hstack([c0_values, dose_values])

# Target: full profiles
y_node = profiles_obs

# Split
X_node_train = X_node[:n_train].astype(np.float32)
y_node_train = y_node[:n_train].astype(np.float32)
X_node_val = X_node[n_train:n_train+n_val].astype(np.float32)
y_node_val = y_node[n_train:n_train+n_val].astype(np.float32)

# Build and train
node_model = NeuralODEModel(hidden_units=64)
node_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss='mse'
)

history_node = node_model.fit(
    X_node_train, y_node_train,
    validation_data=(X_node_val, y_node_val),
    epochs=200,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10, verbose=0)
    ],
    verbose=0
)

print(f"Neural ODE trained for {len(history_node.history['loss'])} epochs")
print(f"Best val loss: {min(history_node.history['val_loss']):.6f}")

Neural ODE trained for 21 epochs
Best val loss: 0.709344


In [9]:
# Visualize Neural ODE predictions vs true profiles
X_node_test = X_node[n_train+n_val:].astype(np.float32)
y_node_test = y_node[n_train+n_val:]

pred_node = node_model.predict(X_node_test, verbose=0)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
test_indices = np.random.choice(len(X_node_test), 6, replace=False)

for idx, ax in zip(test_indices, axes.flat):
    ax.plot(t_sim, y_node_test[idx], 'bo', markersize=3, alpha=0.6, label='Observed')
    ax.plot(t_sim, profiles_true[n_train+n_val+idx], 'g-', linewidth=1, alpha=0.5, label='True')
    ax.plot(t_sim, pred_node[idx], 'r-', linewidth=2, label='Neural ODE')
    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Conc (mg/L)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

plt.suptitle('Neural ODE: Predicted vs Observed Concentration Profiles', y=1.01)
plt.tight_layout()
plt.show()

## 6. PD Modeling — Emax Model

Now we link drug concentration to pharmacological effect using the **sigmoid Emax model**:

$$E = E_0 + \frac{E_{max} \cdot C^\gamma}{EC_{50}^\gamma + C^\gamma}$$

where:
- E0 = baseline effect
- Emax = maximum drug effect
- EC50 = concentration at 50% of Emax
- gamma (Hill coefficient) = steepness of the curve

In [10]:
def sigmoid_emax(c, e0, emax, ec50, gamma):
    """Sigmoid Emax pharmacodynamic model."""
    return e0 + (emax * c**gamma) / (ec50**gamma + c**gamma)

# PD parameters
e0 = 5        # baseline effect
emax = 95     # maximum effect
ec50 = 0.8    # mg/L
gamma = 1.5   # Hill coefficient

# Generate concentration-effect relationship
c_range = np.linspace(0, 3, 200)
effect = sigmoid_emax(c_range, e0, emax, ec50, gamma)

# Effect-time profiles for our simulated population
effect_profiles = sigmoid_emax(profiles_true, e0, emax, ec50, gamma)
effect_profiles_noisy = effect_profiles * (1 + np.random.normal(0, 0.05, effect_profiles.shape))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Concentration-effect curve
axes[0].plot(c_range, effect, 'b-', linewidth=2)
axes[0].axhline(y=e0 + emax/2, color='gray', linestyle='--', alpha=0.5, label='50% Emax')
axes[0].axvline(x=ec50, color='gray', linestyle='--', alpha=0.5, label=f'EC50={ec50}')
axes[0].set_xlabel('Concentration (mg/L)')
axes[0].set_ylabel('Effect (%)')
axes[0].set_title('Sigmoid Emax Model')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Effect-time profiles
for i in range(min(50, n_subjects)):
    axes[1].plot(t_sim, effect_profiles_noisy[i], alpha=0.3, linewidth=0.8)
axes[1].set_xlabel('Time (h)')
axes[1].set_ylabel('Effect (%)')
axes[1].set_title('Effect-Time Profiles')
axes[1].grid(True, alpha=0.3)

# Hysteresis plot (effect vs concentration, showing time)
for i in range(5):
    scatter = axes[2].scatter(profiles_true[i], effect_profiles[i],
                               c=t_sim, cmap='viridis', s=10, alpha=0.7)
axes[2].set_xlabel('Concentration (mg/L)')
axes[2].set_ylabel('Effect (%)')
axes[2].set_title('Concentration-Effect (colored by time)')
plt.colorbar(scatter, ax=axes[2], label='Time (h)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PD Parameters: E0={e0}, Emax={emax}, EC50={ec50}, gamma={gamma}")

PD Parameters: E0=5, Emax=95, EC50=0.8, gamma=1.5


## 7. Full PK/PD Pipeline — End-to-End Deep Learning

We build an end-to-end model that takes a concentration-time profile as input and predicts:
1. **PK parameters** (ka, ke, Vd)
2. **PD parameters** (Emax, EC50, gamma)
3. **Effect-time profile**

This represents a complete neural network-based PK/PD analysis pipeline.

In [11]:
# Generate PD data with inter-individual variability
np.random.seed(123)

# Population PD parameters with IIV
emax_pop, ec50_pop, gamma_pop = 95, 0.8, 1.5
omega_emax, omega_ec50, omega_gamma = 0.1, 0.3, 0.15

pd_params_true = []
effect_profiles_all = []

for i in range(n_subjects):
    emax_i = emax_pop * np.exp(np.random.normal(0, omega_emax))
    ec50_i = ec50_pop * np.exp(np.random.normal(0, omega_ec50))
    gamma_i = gamma_pop * np.exp(np.random.normal(0, omega_gamma))

    eff = sigmoid_emax(profiles_true[i], e0, emax_i, ec50_i, gamma_i)
    eff_noisy = eff * (1 + np.random.normal(0, 0.05, eff.shape))

    pd_params_true.append([emax_i, ec50_i, gamma_i])
    effect_profiles_all.append(eff_noisy)

pd_params_true = np.array(pd_params_true)
effect_profiles_all = np.array(effect_profiles_all)

# Combined targets: PK params (3) + PD params (3) = 6 parameters
all_params = np.hstack([params_true, pd_params_true])
all_params_log = np.log(all_params)

# Build multi-output model
input_layer = keras.layers.Input(shape=(n_timepoints,))
x = keras.layers.Reshape((n_timepoints, 1))(input_layer)

# Shared feature extraction (CNN backbone)
x = keras.layers.Conv1D(32, 5, activation='relu', padding='same')(x)
x = keras.layers.Conv1D(64, 5, activation='relu', padding='same')(x)
x = keras.layers.MaxPooling1D(2)(x)
x = keras.layers.Conv1D(64, 3, activation='relu', padding='same')(x)
x = keras.layers.GlobalAveragePooling1D()(x)
shared = keras.layers.Dense(128, activation='relu')(x)
shared = keras.layers.Dropout(0.2)(shared)

# PK parameter head
pk_branch = keras.layers.Dense(64, activation='relu')(shared)
pk_output = keras.layers.Dense(3, name='pk_params')(pk_branch)

# PD parameter head
pd_branch = keras.layers.Dense(64, activation='relu')(shared)
pd_output = keras.layers.Dense(3, name='pd_params')(pd_branch)

# Effect profile head
eff_branch = keras.layers.Dense(128, activation='relu')(shared)
eff_branch = keras.layers.Dense(128, activation='relu')(eff_branch)
eff_output = keras.layers.Dense(n_timepoints, name='effect_profile')(eff_branch)

model_pkpd = keras.Model(inputs=input_layer,
                          outputs=[pk_output, pd_output, eff_output])

model_pkpd.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        'pk_params': 'mse',
        'pd_params': 'mse',
        'effect_profile': 'mse'
    },
    loss_weights={
        'pk_params': 1.0,
        'pd_params': 1.0,
        'effect_profile': 0.01  # scale down since effect values are larger
    }
)

print("Full PK/PD Model:")
model_pkpd.summary()

Full PK/PD Model:


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 48)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 48, 1)     │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 48, 32)    │        192 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 48, 64)    │     10,304 │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_4[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 24, 64)    │     12,352 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ conv1d_5[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │      8,320 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 128)       │     16,512 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 128)       │     16,512 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pk_params (Dense)   │ (None, 3)         │        195 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pd_params (Dense)   │ (None, 3)         │        195 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ effect_profile      │ (None, 48)        │      6,192 │ dense_10[0][0]    │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 87,286 (340.96 KB)

 Trainable params: 87,286 (340.96 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Prepare targets
y_pk_train = all_params_log[:n_train, :3]
y_pd_train = all_params_log[:n_train, 3:]
y_eff_train = effect_profiles_all[:n_train]

y_pk_val = all_params_log[n_train:n_train+n_val, :3]
y_pd_val = all_params_log[n_train:n_train+n_val, 3:]
y_eff_val = effect_profiles_all[n_train:n_train+n_val]

# Train
history_pkpd = model_pkpd.fit(
    X_train,
    {'pk_params': y_pk_train, 'pd_params': y_pd_train, 'effect_profile': y_eff_train},
    validation_data=(
        X_val,
        {'pk_params': y_pk_val, 'pd_params': y_pd_val, 'effect_profile': y_eff_val}
    ),
    epochs=150,
    batch_size=32,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=8, verbose=0)
    ],
    verbose=0
)

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, key, title in zip(axes,
                            ['pk_params_loss', 'pd_params_loss', 'effect_profile_loss'],
                            ['PK Params Loss', 'PD Params Loss', 'Effect Profile Loss']):
    ax.plot(history_pkpd.history[key], label='Train')
    ax.plot(history_pkpd.history[f'val_{key}'], label='Val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [13]:
# Final evaluation on test set
y_pk_test = all_params_log[n_train+n_val:, :3]
y_pd_test = all_params_log[n_train+n_val:, 3:]
y_eff_test = effect_profiles_all[n_train+n_val:]

pk_pred, pd_pred, eff_pred = model_pkpd.predict(X_test, verbose=0)

# PK parameter evaluation
pk_true_orig = np.exp(y_pk_test)
pk_pred_orig = np.exp(pk_pred)
pd_true_orig = np.exp(y_pd_test)
pd_pred_orig = np.exp(pd_pred)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Top row: PK parameters
for j, name in enumerate(['ka (1/h)', 'ke (1/h)', 'Vd (L)']):
    ax = axes[0, j]
    ax.scatter(pk_true_orig[:, j], pk_pred_orig[:, j], alpha=0.5, s=20)
    lims = [min(pk_true_orig[:, j].min(), pk_pred_orig[:, j].min()) * 0.9,
            max(pk_true_orig[:, j].max(), pk_pred_orig[:, j].max()) * 1.1]
    ax.plot(lims, lims, 'r--', linewidth=1.5)
    ax.set_xlabel(f'True {name}')
    ax.set_ylabel(f'Predicted {name}')
    ax.set_title(f'PK: {name}')
    r2 = 1 - np.sum((pk_true_orig[:, j] - pk_pred_orig[:, j])**2) / \
             np.sum((pk_true_orig[:, j] - pk_true_orig[:, j].mean())**2)
    ax.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.grid(True, alpha=0.3)

# Bottom row: PD parameters
for j, name in enumerate(['Emax', 'EC50 (mg/L)', 'gamma']):
    ax = axes[1, j]
    ax.scatter(pd_true_orig[:, j], pd_pred_orig[:, j], alpha=0.5, s=20, color='green')
    lims = [min(pd_true_orig[:, j].min(), pd_pred_orig[:, j].min()) * 0.9,
            max(pd_true_orig[:, j].max(), pd_pred_orig[:, j].max()) * 1.1]
    ax.plot(lims, lims, 'r--', linewidth=1.5)
    ax.set_xlabel(f'True {name}')
    ax.set_ylabel(f'Predicted {name}')
    ax.set_title(f'PD: {name}')
    r2 = 1 - np.sum((pd_true_orig[:, j] - pd_pred_orig[:, j])**2) / \
             np.sum((pd_true_orig[:, j] - pd_true_orig[:, j].mean())**2)
    ax.text(0.05, 0.95, f'R² = {r2:.3f}', transform=ax.transAxes, va='top',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
    ax.grid(True, alpha=0.3)

plt.suptitle('Full PK/PD Model — Parameter Estimation (Test Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [14]:
# Compare predicted vs true effect-time profiles
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
test_indices = np.random.choice(len(X_test), 8, replace=False)

for idx, ax in zip(test_indices, axes.flat):
    ax.plot(t_sim, y_eff_test[idx], 'b-', linewidth=1, alpha=0.7, label='True Effect')
    ax.plot(t_sim, eff_pred[idx], 'r--', linewidth=2, label='Predicted Effect')
    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Effect (%)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)

plt.suptitle('Effect-Time Profile Predictions (Test Set)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated PK/PD modeling using both classical and deep learning approaches:

| Approach | Method | Strengths | Limitations |
|----------|--------|-----------|-------------|
| **Classical ODE** | Analytical/numerical solutions | Mechanistically interpretable, gold standard | Requires model specification, slow for large populations |
| **NN Parameter Estimation** | CNN predicts PK params from profiles | Fast inference, handles noise | Requires labeled training data |
| **Neural ODE** | NN learns dC/dt dynamics | Model-free, flexible | Less interpretable, needs more data |
| **Full PK/PD Pipeline** | Multi-output CNN | End-to-end, simultaneous PK+PD | Complex training, data-hungry |

### Key Takeaways
- Deep learning can complement traditional PK/PD modeling, especially for rapid parameter estimation
- Neural ODEs offer a flexible alternative when the structural model is unknown
- Multi-output architectures enable joint PK/PD analysis from concentration data alone
- Population variability (IIV) is naturally handled by training on diverse synthetic patients

### References
- Pharmacokinetics: Rowland & Tozer, "Clinical Pharmacokinetics and Pharmacodynamics"
- Neural ODEs: Chen et al., "Neural Ordinary Differential Equations" (NeurIPS 2018)
- ML in Pharmacometrics: Lu et al., "Neural network-based pharmacokinetic predictions" (CPT:PSP, 2021)